# Tile Extraction from Mosaic
Read saved mosaic images → sample random tiles → filter by tissue content → balance by class → save.

## Spatial Train/Val Split
Validation tiles are chosen first. All training tiles are then sampled only from
positions that have **zero pixel overlap** with any validation tile. This guarantees
the validation set is a true hold-out with no shared pixels.

In [ ]:
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colors as mcolors
import os

In [ ]:
# Paths
MOSAIC_DIR = Path('annotations/big_tile')
OUT_DIR    = MOSAIC_DIR / 'tiles'
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'images').mkdir(exist_ok=True)
(OUT_DIR / 'masks').mkdir(exist_ok=True)

# Tile parameters
TILE_SIZE          = 1024    # px
MIN_TISSUE_FRAC    = 0.3     # min fraction of tile pixels that must be non-background (mask > 0)
TARGET_TILES       = 2000     # training tiles to collect
N_VAL_TILES        = 20      # validation tiles (spatially held out)
MAX_ATTEMPTS       = 50_000  # safety cap on random samples
SEED               = 67

# Class balancing
# Oversample tiles containing underrepresented classes by this multiplier
# Set to 1.0 to disable balancing
BALANCE_OVERSAMPLE = 3.0    # tiles with rare classes are kept up to this many extra times
RARE_CLASS_THRESH  = 0.05   # a class is "rare" if it covers < this fraction of all tissue pixels

# Label map (must match what was used to build the mosaic)
PRIORITY_ORDER = [
    "Whitespace",
    "Vasculature",
    "Muscle",
    "Nerve",
    "Fat",
    "TLS",
    "PDAC",
    "Duct",
    "Islet",
    "ECM",
]
label_to_class = {lbl: len(PRIORITY_ORDER) - i for i, lbl in enumerate(PRIORITY_ORDER)}
class_to_label = {v: k for k, v in label_to_class.items()}
NUM_CLASSES    = len(PRIORITY_ORDER) + 1   # +1 for background = 0

In [ ]:
# Load mosaics
Image.MAX_IMAGE_PIXELS = None  # disable PIL decompression bomb warning for large images

he_mosaic   = np.array(Image.open(MOSAIC_DIR / 'mosaic_he.png').convert('RGB'))
mask_mosaic = np.array(Image.open(MOSAIC_DIR / 'mosaic_mask.png'))

H, W = mask_mosaic.shape
print(f"Mosaic size  : {W} × {H} px")
print(f"Unique classes in mask: {np.unique(mask_mosaic).tolist()}")

In [ ]:
# Compute class frequencies to identify rare classes
tissue_pixels = mask_mosaic[mask_mosaic > 0]
class_counts  = {c: int((tissue_pixels == c).sum()) for c in np.unique(tissue_pixels)}
total_tissue  = tissue_pixels.size

print("Class distribution (tissue pixels only):")
for cls, cnt in sorted(class_counts.items()):
    lbl  = class_to_label.get(cls, f"cls{cls}")
    frac = cnt / total_tissue if total_tissue > 0 else 0
    print(f"  {cls:2d}  {lbl:<15}  {cnt:8d}  ({frac*100:.2f}%)")

rare_classes = {c for c, cnt in class_counts.items()
                if cnt / total_tissue < RARE_CLASS_THRESH}
print(f"\nRare classes (< {RARE_CLASS_THRESH*100:.0f}% of tissue): "
      f"{[class_to_label.get(c, c) for c in rare_classes]}")

In [ ]:
# Helper: check if a candidate position overlaps any existing tile footprint
def overlaps_any(x, y, tile_size, existing_positions):
    """Return True if the tile at (x,y) overlaps with any tile in existing_positions."""
    for ex, ey in existing_positions:
        # Two TILE_SIZE squares overlap iff their 1-D intervals overlap on both axes
        if (x < ex + tile_size and x + tile_size > ex and
                y < ey + tile_size and y + tile_size > ey):
            return True
    return False

rng = np.random.default_rng(SEED)

# Step 1: Sample validation tiles first
val_he, val_mask, val_meta = [], [], []
val_positions = []   # (x, y) of each accepted val tile
attempt_val = 0

while len(val_he) < N_VAL_TILES and attempt_val < MAX_ATTEMPTS:
    attempt_val += 1
    x = int(rng.integers(0, W - TILE_SIZE))
    y = int(rng.integers(0, H - TILE_SIZE))

    mask_tile = mask_mosaic[y:y+TILE_SIZE, x:x+TILE_SIZE]
    if (mask_tile > 0).mean() < MIN_TISSUE_FRAC:
        continue
    if overlaps_any(x, y, TILE_SIZE, val_positions):
        continue

    classes_present = set(np.unique(mask_tile[mask_tile > 0]).tolist())
    val_he.append(he_mosaic[y:y+TILE_SIZE, x:x+TILE_SIZE])
    val_mask.append(mask_tile)
    val_meta.append((x, y, (mask_tile > 0).mean(), classes_present))
    val_positions.append((x, y))

print(f"Validation tiles collected: {len(val_he)} in {attempt_val} attempts")

# Step 2: Sample training tiles — no overlap with val positions
tiles_he, tiles_mask, tile_meta = [], [], []
class_tile_counts = {c: 0 for c in class_counts}
attempt_train = 0
n_collected   = 0

while n_collected < TARGET_TILES and attempt_train < MAX_ATTEMPTS:
    attempt_train += 1
    x = int(rng.integers(0, W - TILE_SIZE))
    y = int(rng.integers(0, H - TILE_SIZE))

    mask_tile = mask_mosaic[y:y+TILE_SIZE, x:x+TILE_SIZE]
    tissue_frac = (mask_tile > 0).mean()
    if tissue_frac < MIN_TISSUE_FRAC:
        continue

    # Reject if this position overlaps any validation tile
    if overlaps_any(x, y, TILE_SIZE, val_positions):
        continue

    classes_present = set(np.unique(mask_tile[mask_tile > 0]).tolist())
    has_rare  = bool(classes_present & rare_classes)
    max_copies = int(BALANCE_OVERSAMPLE) if has_rare else 1
    existing   = sum(1 for m in tile_meta if (m[0], m[1]) == (x, y))
    if existing >= max_copies:
        continue

    he_tile = he_mosaic[y:y+TILE_SIZE, x:x+TILE_SIZE]
    tiles_he.append(he_tile)
    tiles_mask.append(mask_tile)
    tile_meta.append((x, y, tissue_frac, classes_present))
    n_collected += 1

    for c in classes_present:
        if c in class_tile_counts:
            class_tile_counts[c] += 1

print(f"\nTraining tiles: {n_collected} collected in {attempt_train} attempts "
      f"({attempt_train/max(n_collected,1):.1f} attempts/tile)")
print(f"\nTiles per class (train):")
for cls, cnt in sorted(class_tile_counts.items()):
    print(f"  {cls:2d}  {class_to_label.get(cls,'?'):<15}  {cnt} tiles")

In [ ]:
# Visualise tile locations on the mosaic
DISPLAY_SCALE = 0.05   # fraction of original mosaic size to display (tune as needed)

# Downscale the mask mosaic for display
disp_h = max(1, int(H * DISPLAY_SCALE))
disp_w = max(1, int(W * DISPLAY_SCALE))

mask_disp = cv2.resize(mask_mosaic, (disp_w, disp_h), interpolation=cv2.INTER_NEAREST)

# Colour the mask using the same colormap
cmap_base_v   = plt.cm.get_cmap("tab10", NUM_CLASSES)
colors_rgba_v = cmap_base_v(np.arange(NUM_CLASSES))
colors_rgba_v[0] = [0, 0, 0, 1]
tile_cmap_v   = mcolors.ListedColormap(colors_rgba_v)

fig, ax = plt.subplots(1, 1, figsize=(16, 16 * disp_h / disp_w))
ax.imshow(mask_disp, cmap=tile_cmap_v, vmin=0, vmax=NUM_CLASSES - 1,
          interpolation='nearest')

# Draw training tile boxes
for x, y, tf, _ in tile_meta:
    rx = x * DISPLAY_SCALE
    ry = y * DISPLAY_SCALE
    rw = TILE_SIZE * DISPLAY_SCALE
    rh = TILE_SIZE * DISPLAY_SCALE
    rect = mpatches.Rectangle((rx, ry), rw, rh,
                               linewidth=0.4, edgecolor='cyan',
                               facecolor='blue', alpha=0.2)
    ax.add_patch(rect)

# Draw validation tile boxes
for x, y, tf, _ in val_meta:
    rx = x * DISPLAY_SCALE
    ry = y * DISPLAY_SCALE
    rw = TILE_SIZE * DISPLAY_SCALE
    rh = TILE_SIZE * DISPLAY_SCALE
    rect = mpatches.Rectangle((rx, ry), rw, rh,
                               linewidth=1.2, edgecolor='red',
                               facecolor='red', alpha=0.75)
    ax.add_patch(rect)
    # Bold border on top
    rect2 = mpatches.Rectangle((rx, ry), rw, rh,
                                linewidth=1.2, edgecolor='red',
                                facecolor='none')
    ax.add_patch(rect2)

# Legend
legend_handles = [mpatches.Patch(color=tile_cmap_v(0), label="0: background")]
for lbl, idx_ in sorted(label_to_class.items(), key=lambda kv: kv[1]):
    legend_handles.append(mpatches.Patch(color=tile_cmap_v(idx_), label=f"{idx_}: {lbl}"))
legend_handles += [
    mpatches.Patch(edgecolor='cyan',  facecolor='none', label=f'Train tiles (n={len(tile_meta)})'),
    mpatches.Patch(edgecolor='red',   facecolor='red',  alpha=0.75,
                   label=f'Val tiles   (n={len(val_meta)})'),
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=7,
          framealpha=0.8, ncol=2)

ax.set_title(f"Tile locations on mosaic  (display scale={DISPLAY_SCALE:.2f}×)", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Preview a random selection of TRAINING tiles
N_PREVIEW = min(8, n_collected)
preview_idx = rng.choice(n_collected, N_PREVIEW, replace=False)

cmap_base   = plt.cm.get_cmap("tab10", NUM_CLASSES)
colors_rgba = cmap_base(np.arange(NUM_CLASSES))
colors_rgba[0] = [0, 0, 0, 1]
tile_cmap   = mcolors.ListedColormap(colors_rgba)

fig, axes = plt.subplots(N_PREVIEW, 2, figsize=(10, 5 * N_PREVIEW))
if N_PREVIEW == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(preview_idx):
    x, y, tf, cls_present = tile_meta[idx]
    axes[row, 0].imshow(tiles_he[idx])
    axes[row, 0].set_title(f"H&E  ({x},{y})  tissue={tf*100:.1f}%", fontsize=8)
    axes[row, 0].axis('off')
    axes[row, 1].imshow(tiles_mask[idx], cmap=tile_cmap, vmin=0, vmax=NUM_CLASSES-1)
    axes[row, 1].set_title(f"Mask  classes={sorted(cls_present)}", fontsize=8)
    axes[row, 1].axis('off')

legend_handles = [mpatches.Patch(color=tile_cmap(0), label="0: background")]
for lbl, idx_ in sorted(label_to_class.items(), key=lambda x: x[1]):
    legend_handles.append(mpatches.Patch(color=tile_cmap(idx_), label=f"{idx_}: {lbl}"))
fig.legend(handles=legend_handles, loc='lower center', ncol=5, fontsize=8, frameon=True)
plt.suptitle("Training tiles preview", y=1.001)
plt.tight_layout(); plt.show()

# Preview validation tiles
N_VAL_PREVIEW = min(4, len(val_he))
val_preview_idx = rng.choice(len(val_he), N_VAL_PREVIEW, replace=False)

fig2, axes2 = plt.subplots(N_VAL_PREVIEW, 2, figsize=(10, 5 * N_VAL_PREVIEW))
if N_VAL_PREVIEW == 1:
    axes2 = axes2[np.newaxis, :]

for row, idx in enumerate(val_preview_idx):
    x, y, tf, cls_present = val_meta[idx]
    axes2[row, 0].imshow(val_he[idx])
    axes2[row, 0].set_title(f"[VAL] H&E  ({x},{y})  tissue={tf*100:.1f}%", fontsize=8)
    axes2[row, 0].axis('off')
    axes2[row, 1].imshow(val_mask[idx], cmap=tile_cmap, vmin=0, vmax=NUM_CLASSES-1)
    axes2[row, 1].set_title(f"[VAL] Mask  classes={sorted(cls_present)}", fontsize=8)
    axes2[row, 1].axis('off')

fig2.legend(handles=legend_handles, loc='lower center', ncol=5, fontsize=8, frameon=True)
plt.suptitle("Validation tiles preview", y=1.001)
plt.tight_layout(); plt.show()

In [ ]:
# Save training tiles
he_stack_train   = np.stack(tiles_he)
mask_stack_train = np.stack(tiles_mask)
meta_train       = np.array(
    [(x, y, tf) for x, y, tf, _ in tile_meta],
    dtype=[('x', np.int32), ('y', np.int32), ('tissue_frac', np.float32)],
)
np.savez_compressed(
    OUT_DIR / 'tiles_train.npz',
    he=he_stack_train, masks=mask_stack_train, meta=meta_train,
)
print(f"Saved {len(tiles_he)} training tile pairs → {OUT_DIR / 'tiles_train.npz'}")

# Save validation tiles
he_stack_val   = np.stack(val_he)
mask_stack_val = np.stack(val_mask)
meta_val       = np.array(
    [(x, y, tf) for x, y, tf, _ in val_meta],
    dtype=[('x', np.int32), ('y', np.int32), ('tissue_frac', np.float32)],
)
np.savez_compressed(
    OUT_DIR / 'tiles_val.npz',
    he=he_stack_val, masks=mask_stack_val, meta=meta_val,
)
print(f"Saved {len(val_he)} validation tile pairs  → {OUT_DIR / 'tiles_val.npz'}")

In [ ]:
# Quick load check
data_tr = np.load(OUT_DIR / 'tiles_train.npz')
data_va = np.load(OUT_DIR / 'tiles_val.npz')
print(f"\nTrain he shape : {data_tr['he'].shape},  mask shape: {data_tr['masks'].shape}")
print(f"Val   he shape : {data_va['he'].shape},  mask shape: {data_va['masks'].shape}")